# Forward Motors: Stage 3 Data Collection and Analysis

**Auction price data track**

DSBA 6165 AI and Deep Learning | Mam Salan Njie and Will Moore

This notebook covers the auction price dataset, which is my half of the project. Will's notebook
covers the CarDD damage images and the photos we collect at the auction, so image data is not
analyzed here.

The project question this data supports: MMR is the wholesale price guide dealers rely on, and it
is accurate on average but can miss badly on an individual car. We want to predict where it will
miss before the bidding starts. That makes `sellingprice` the target and `mmr` an input, and it
makes the gap between the two the thing worth studying.

**What's in this notebook**

1. Loading the data and a first look
2. Missing values and how I handle them
3. The target variable
4. Numeric features
5. Categorical features
6. MMR as a benchmark, which is the core of our problem
7. Error by consignor
8. Time structure, since we split by date
9. Cleaning decisions
10. Train, validation and test split
11. What this data cannot tell us
12. Dataset at a glance
13. What this means for our three research questions
14. Summary and what this means for modeling

## 1. Loading the data

The dataset is "Vehicle Sales Data" by Syed Anwar Afridi on Kaggle. It records wholesale auction
sales, one row per vehicle sold.

The file is too large to commit to GitHub, so the notebook looks for it in three places: the
current folder, a Google Drive path, and finally a Kaggle download. Put `car_prices.csv` beside
this notebook or in Drive and the first option works.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED)

In [ ]:
CSV_NAME = "car_prices.csv"

candidates = [
    CSV_NAME,
    "data/" + CSV_NAME,
    "/content/" + CSV_NAME,
    "/content/drive/MyDrive/forward-motors/" + CSV_NAME,
]

path = next((p for p in candidates if os.path.exists(p)), None)

if path is None:
    # Fall back to downloading from Kaggle. Needs a kaggle.json token in Colab.
    import kagglehub
    folder = kagglehub.dataset_download("syedanwarafridi/vehicle-sales-data")
    path = os.path.join(folder, CSV_NAME)

print("Loading from:", path)
cars = pd.read_csv(path)
print("Rows:", f"{len(cars):,}", " Columns:", cars.shape[1])
cars.head()

In [ ]:
cars.info()

### What each column is

| Column | Meaning |
|---|---|
| `year` | Model year of the vehicle |
| `make`, `model`, `trim`, `body` | What the vehicle is |
| `transmission` | Automatic or manual |
| `vin` | Vehicle identification number, unique per physical car |
| `state` | State where the sale happened |
| `condition` | Inspection grade assigned before the sale |
| `odometer` | Mileage at sale |
| `color`, `interior` | Exterior and interior color |
| `seller` | The company consigning the vehicle, usually a lender or fleet |
| `mmr` | Manheim Market Report value, the industry price guide |
| `sellingprice` | What the vehicle actually sold for, our target |
| `saledate` | Date and time of the sale |

## 2. Missing values

Before anything else I need to know what's missing and whether it's missing in a way that matters.

In [ ]:
missing = pd.DataFrame({
    "missing": cars.isna().sum(),
    "percent": (cars.isna().mean() * 100).round(3)
}).sort_values("missing", ascending=False)

display(missing)

plt.figure(figsize=(9, 5))
subset = missing[missing["missing"] > 0]
plt.barh(subset.index[::-1], subset["percent"][::-1], color="#4C72B0")
plt.xlabel("Percent missing")
plt.title("Missing values by column")
plt.tight_layout()
plt.show()

Three groups here, and they need different treatment.

`transmission` is missing on a large share of rows. That's too many to drop, and transmission is
not central to price for most of these vehicles, so I treat the blanks as their own category called
"unknown" rather than throwing away the rows.

The vehicle description fields (`make`, `model`, `trim`, `body`) are each missing on a small
percentage, and they tend to be missing together on the same rows. A row with no make or model
tells us almost nothing about the car, so those rows get dropped.

`sellingprice`, `mmr` and `saledate` are missing on only a handful of rows. Since `sellingprice` is
the target and `mmr` is the benchmark we are measuring against, a row without them is unusable and
is dropped.

In [ ]:
# Are the description fields missing on the same rows?
desc_cols = ["make", "model", "trim", "body"]
missing_any = cars[desc_cols].isna().any(axis=1).sum()
missing_all = cars[desc_cols].isna().all(axis=1).sum()

print(f"Rows missing at least one description field: {missing_any:,}")
print(f"Rows missing all four description fields:    {missing_all:,}")
print(f"So {missing_all/max(missing_any,1):.1%} of the affected rows are missing all four at once.")

## 3. The target variable

`sellingprice` is the amount the vehicle sold for in dollars. This is a regression problem, so
there are no classes. The distribution still matters, because a heavy right tail changes which
error metric is honest.

In [ ]:
price = cars["sellingprice"].dropna()

print(price.describe().round(0).to_string())
print()
for q in [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]:
    print(f"  {q:>6.1%} percentile: ${price.quantile(q):>10,.0f}")
print(f"\nSkewness: {price.skew():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(price, bins=80, color="#4C72B0", edgecolor="white")
axes[0].set_title("Selling price")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Vehicles")

axes[1].hist(np.log10(price[price > 0]), bins=80, color="#55A868", edgecolor="white")
axes[1].set_title("Selling price, log scale")
axes[1].set_xlabel("log10(price)")

axes[2].boxplot(price, vert=True, widths=0.5)
axes[2].set_title("Selling price spread")
axes[2].set_ylabel("Price ($)")

plt.tight_layout()
plt.show()

The distribution is right skewed. Most vehicles sell in the low thousands, with a long tail of
expensive cars running far above that.

Two consequences for modeling. Average dollar error across all vehicles will be dominated by the
expensive end, so I report percentage error alongside it and break error out by price band. And
because the tail is long rather than symmetric, a model trained on raw dollars will spend most of
its capacity on the few expensive cars. Predicting log price is a reasonable alternative to test in
the modeling stage.

## 4. Numeric features

In [ ]:
num_cols = ["year", "condition", "odometer", "mmr", "sellingprice"]
cars[num_cols].describe().T.round(2)

One thing to note before reading the charts: `condition` is not a 1 to 5 grade. It runs from 1 to 49
with a mean near 31, so it is a finer scale than the five-point grade dealers usually talk about.
The odometer column also has a maximum of 999,999, which is a placeholder rather than a real
reading, and section 9 removes those rows.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.ravel(), ["year", "condition", "odometer", "mmr"]):
    ax.hist(cars[col].dropna(), bins=60, color="#4C72B0", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel(col)
plt.suptitle("Numeric feature distributions", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(cars[num_cols].corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation between numeric features")
plt.tight_layout()
plt.show()

In [ ]:
# Relationship between the main numeric drivers and price.
plot_pool = cars.dropna(subset=["sellingprice", "odometer", "condition", "mmr"])
sample = plot_pool.sample(min(20000, len(plot_pool)), random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(sample["odometer"], sample["sellingprice"], s=3, alpha=0.15, color="#4C72B0")
axes[0].set_xlabel("Odometer (miles)")
axes[0].set_ylabel("Selling price ($)")
axes[0].set_title("Mileage vs price")

axes[1].scatter(sample["condition"], sample["sellingprice"], s=3, alpha=0.15, color="#55A868")
axes[1].set_xlabel("Condition grade")
axes[1].set_title("Condition vs price")

axes[2].scatter(sample["mmr"], sample["sellingprice"], s=3, alpha=0.15, color="#C44E52")
lims = [0, sample[["mmr", "sellingprice"]].quantile(0.999).max()]
axes[2].plot(lims, lims, "k--", linewidth=1, label="sold exactly at MMR")
axes[2].set_xlim(lims); axes[2].set_ylim(lims)
axes[2].set_xlabel("MMR ($)")
axes[2].set_title("MMR vs price")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Price by model year, which captures depreciation.
by_year = cars.groupby("year")["sellingprice"].agg(["median", "count"])
by_year = by_year[by_year["count"] >= 50]

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(by_year.index, by_year["median"], marker="o", color="#4C72B0")
ax1.set_xlabel("Model year")
ax1.set_ylabel("Median selling price ($)", color="#4C72B0")
ax2 = ax1.twinx()
ax2.bar(by_year.index, by_year["count"], alpha=0.2, color="#937860")
ax2.set_ylabel("Vehicles sold", color="#937860")
ax2.grid(False)
plt.title("Median price and volume by model year")
plt.tight_layout()
plt.show()

Mileage and price move in opposite directions as expected, but the relationship is clearly not a
straight line. Price falls steeply over the first stretch of miles and then flattens out. A linear
model would have to be told about that bend, while a neural network can learn it, which is part of
the argument for our approach.

MMR and selling price line up closely around the diagonal, which is what we would expect from a
price guide built on auction sales. The spread around that diagonal is the entire opportunity in
this project. Section 6 measures it.

## 5. Categorical features

These matter for encoding decisions. A column with ten values can be one-hot encoded. A column with
thousands cannot, and needs an embedding or a frequency-based grouping instead.

In [ ]:
cat_cols = ["make", "model", "trim", "body", "transmission", "state",
            "color", "interior", "seller"]

card = pd.DataFrame({
    "unique_values": [cars[c].nunique() for c in cat_cols],
    "most_common": [cars[c].mode()[0] if cars[c].notna().any() else None for c in cat_cols],
    "top_share": [(cars[c].value_counts(normalize=True).iloc[0] * 100).round(1)
                  for c in cat_cols],
}, index=cat_cols).sort_values("unique_values", ascending=False)

card

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, col in zip(axes.ravel(), ["make", "body", "state", "transmission"]):
    top = cars[col].value_counts().head(12)
    ax.barh(top.index[::-1], top.values[::-1], color="#4C72B0")
    ax.set_title(f"Most common {col}")
    ax.set_xlabel("Vehicles")
plt.suptitle("Categorical feature distributions (top 12 each)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Does price vary across the big categorical groups?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_makes = cars["make"].value_counts().head(10).index
sns.boxplot(data=cars[cars["make"].isin(top_makes)], x="make", y="sellingprice",
            ax=axes[0], showfliers=False, color="#4C72B0")
axes[0].set_title("Price by make (top 10 by volume)")
axes[0].tick_params(axis="x", rotation=45)

top_bodies = cars["body"].value_counts().head(8).index
sns.boxplot(data=cars[cars["body"].isin(top_bodies)], x="body", y="sellingprice",
            ax=axes[1], showfliers=False, color="#55A868")
axes[1].set_title("Price by body style (top 8 by volume)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

The cardinality table drives the encoding plan. `body`, `transmission` and `state` are small enough
to one-hot encode directly. `make` is borderline. `model`, `trim` and `seller` have far too many
distinct values for one-hot encoding, since that would add thousands of mostly empty columns.

For those I plan to use learned embeddings in the TensorFlow model, which map each category to a
small dense vector the network trains along with everything else. This is the same idea as word
embeddings and it's one of the places a neural network has a real advantage over one-hot encoding
plus a linear model.

`seller` is worth keeping despite its cardinality. Different consignors have different reputations
and reserve behavior, so who is selling the car plausibly carries price signal that MMR does not
capture.

## 6. MMR as a benchmark

This is the section that defines the project. MMR is already a price estimate, so any model we
build has to be measured against it rather than against nothing. Here I quantify how far off MMR is
on individual vehicles.

In [ ]:
benchmark = cars.dropna(subset=["sellingprice", "mmr"]).copy()
benchmark["error"] = benchmark["sellingprice"] - benchmark["mmr"]
benchmark["abs_error"] = benchmark["error"].abs()
benchmark["pct_error"] = benchmark["error"] / benchmark["mmr"].replace(0, np.nan) * 100

mae = benchmark["abs_error"].mean()
medae = benchmark["abs_error"].median()
bias = benchmark["error"].mean()
ratio = (benchmark["sellingprice"].sum() / benchmark["mmr"].sum()) * 100

print(f"Vehicles compared:               {len(benchmark):,}")
print(f"MMR mean absolute error:         ${mae:,.0f}")
print(f"MMR median absolute error:       ${medae:,.0f}")
print(f"Mean signed error (bias):        ${bias:,.0f}")
print(f"Total sold / total MMR:          {ratio:.1f}%")
print()
print(f"Sold ABOVE MMR:  {(benchmark['error'] > 0).mean():.1%}")
print(f"Sold BELOW MMR:  {(benchmark['error'] < 0).mean():.1%}")
print()
for t in [500, 1000, 2000, 5000]:
    print(f"Off by more than ${t:,}: {(benchmark['abs_error'] > t).mean():.1%} of vehicles")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

clip = benchmark["error"].quantile([0.005, 0.995])
axes[0].hist(benchmark["error"].clip(*clip), bins=100, color="#4C72B0", edgecolor="white")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[0].set_xlabel("Selling price minus MMR ($)")
axes[0].set_ylabel("Vehicles")
axes[0].set_title("MMR is centered, but wide")

pct_clip = benchmark["pct_error"].quantile([0.005, 0.995])
axes[1].hist(benchmark["pct_error"].clip(*pct_clip), bins=100, color="#C44E52",
             edgecolor="white")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Error as percent of MMR")
axes[1].set_title("Same error, as a percentage")

plt.tight_layout()
plt.show()

In [ ]:
# Does MMR miss more on some kinds of vehicle than others?
benchmark["price_band"] = pd.qcut(benchmark["mmr"], 5,
                                  labels=["cheapest 20%", "low", "mid", "high", "priciest 20%"])

band = benchmark.groupby("price_band", observed=True).agg(
    vehicles=("abs_error", "size"),
    mean_abs_error=("abs_error", "mean"),
    median_mmr=("mmr", "median"),
)
band["error_as_pct_of_mmr"] = (band["mean_abs_error"] / band["median_mmr"] * 100).round(1)
display(band.round(0))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(band.index.astype(str), band["mean_abs_error"], color="#4C72B0")
axes[0].set_ylabel("Mean absolute error ($)")
axes[0].set_title("MMR error in dollars, by price band")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(band.index.astype(str), band["error_as_pct_of_mmr"], color="#C44E52")
axes[1].set_ylabel("Error as percent of MMR")
axes[1].set_title("The same error, relative to the car's value")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Error by condition grade: is MMR worse on rough cars?
cond = benchmark.dropna(subset=["condition"]).copy()
cond["condition_bin"] = pd.cut(cond["condition"], bins=5)
by_cond = cond.groupby("condition_bin", observed=True)["abs_error"].agg(["mean", "size"])

plt.figure(figsize=(9, 4.5))
plt.bar(by_cond.index.astype(str), by_cond["mean"], color="#55A868")
plt.ylabel("Mean absolute error ($)")
plt.xlabel("Condition grade")
plt.title("MMR error by condition grade")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display(by_cond.round(0))

This is the central finding of the data analysis, and it confirms the premise of the project.

In aggregate MMR looks excellent. Total sales come to 98.9 percent of total MMR value and the mean
signed error is only about 158 dollars below guide, so on average the guide is doing what Manheim
says it does.

Per vehicle the picture is different. Mean absolute error is 1,087 dollars and the median is 700.
Almost 61 percent of vehicles miss their guide value by more than 500 dollars, 36.6 percent miss by
more than 1,000, and 13.5 percent miss by more than 2,000. Profit per retail unit at a large dealer
runs in the low hundreds, so a 1,000 dollar miss is the difference between a good buy and a loss,
and more than a third of the cars in this data are in that territory.

Our proposal put a number on that. CarMax earned about 317 dollars of profit per retail unit in
FY2026. The average MMR miss measured here is 1,087 dollars, more than three times that figure. So
the typical error in the guide a buyer prices from is larger than the entire margin on the car. That
is the gap this project exists to narrow, and it is now measured on real sales rather than assumed.

The error is also uneven. In dollars it climbs with the value of the car, from 786 on the cheapest
fifth to 1,526 on the priciest. As a share of the car's value it runs the other way, from 26 percent
on the cheapest fifth down to 6 percent on the priciest. So the guide is proportionally least
reliable on inexpensive cars, even though the biggest dollar misses are on expensive ones. Which
view matters depends on whether a buyer thinks in dollars or in margin percentage.

The condition breakdown points the same direction and connects to Will's half of the project. Mean
error falls from about 1,396 dollars on the roughest cars to 930 on cars in good condition. MMR is
least reliable exactly where condition is worst, which is the case a damage detector reading
photographs would be built to catch.

## 7. Error by consignor

The dataset has no information about the dealer doing the buying, but it does identify the
consignor putting each car into the auction. That is the one seller-side field available, so it is
worth checking whether MMR is more reliable for some consignors than others.

In [ ]:
# Does MMR miss more on cars from some consignors than others?
# Only sellers with enough volume to be meaningful.
seller_stats = benchmark.groupby("seller").agg(
    vehicles=("abs_error", "size"),
    mean_abs_error=("abs_error", "mean"),
    mean_signed_error=("error", "mean"),
    median_mmr=("mmr", "median"),
)
big_sellers = seller_stats[seller_stats["vehicles"] >= 200].copy()
big_sellers["error_pct_of_mmr"] = (big_sellers["mean_abs_error"]
                                   / big_sellers["median_mmr"] * 100)

print(f"Sellers in the data:              {seller_stats.shape[0]:,}")
print(f"Sellers with 200+ sales:          {len(big_sellers):,}")
print(f"Share of all sales they account for: "
      f"{big_sellers['vehicles'].sum() / len(benchmark):.1%}")
print()
print("Spread in MMR error across these sellers:")
print(f"  lowest  mean abs error: ${big_sellers['mean_abs_error'].min():,.0f}")
print(f"  median  mean abs error: ${big_sellers['mean_abs_error'].median():,.0f}")
print(f"  highest mean abs error: ${big_sellers['mean_abs_error'].max():,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].hist(big_sellers["mean_abs_error"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].axvline(benchmark["abs_error"].mean(), color="black", linestyle="--",
                linewidth=1.5, label="overall average")
axes[0].set_xlabel("Mean absolute MMR error ($)")
axes[0].set_ylabel("Sellers")
axes[0].set_title("MMR error varies by consignor")
axes[0].legend()

axes[1].hist(big_sellers["mean_signed_error"], bins=40, color="#55A868",
             edgecolor="white")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Mean signed error ($)")
axes[1].set_title("Some consignors sell above MMR, others below")

plt.tight_layout()
plt.show()

print("Ten consignors whose cars sell furthest above MMR on average:")
display(big_sellers.nlargest(10, "mean_signed_error")[
    ["vehicles", "mean_signed_error", "mean_abs_error", "median_mmr"]].round(0))

print("Ten whose cars sell furthest below:")
display(big_sellers.nsmallest(10, "mean_signed_error")[
    ["vehicles", "mean_signed_error", "mean_abs_error", "median_mmr"]].round(0))

This is the closest thing in the data to information about who is behind the sale, and it looks
useful. The consignor is the company putting the car into the auction, usually a lender, a rental
company or a fleet. There is no buyer information in this dataset, so this is a one-sided view.

The signed error spread is the part worth paying attention to. If MMR were equally accurate across
consignors these averages would sit near zero. They do not. Across the 313 consignors with at least
200 sales, which together account for about 70 percent of all sales, average absolute error runs
from 329 dollars at the best to 6,961 at the worst, against an overall average of 1,087.

The names at the bottom of the signed-error list are what make this interesting. Rental and fleet
operators and captive finance arms, including hertz, two enterprise entities, mercedes-benz usa and
td auto finance, all sell systematically below their guide values, in one case by nearly 7,000
dollars on average. Independent dealers sit at the other end, selling above guide. Whatever is
driving that, reserve policy, vehicle history, or how the cars are presented, MMR is not capturing
it, and it is visible from the seller field alone.

That is a concrete example of the signal our first research question is looking for, and it means
`seller` is worth keeping as a model input even though its cardinality rules out one-hot encoding.

One caution: a consignor's average can be a stand-in for the kind of car it sells rather than
anything about the seller itself. A company that only moves late-model lease returns will look
different from one selling older repossessions. Whether the seller adds anything beyond the vehicle
attributes already in the data is a modeling question, not something this chart settles.

## 8. Time structure

Our split is by date rather than random, so I need to understand how the data is distributed over
time before splitting.

In [ ]:
# saledate looks like "Tue Dec 16 2014 12:30:00 GMT-0800 (PST)".
# Taking the first 4 fields gives a clean date to parse.
dated = cars.dropna(subset=["saledate"]).copy()
dated["sale_dt"] = pd.to_datetime(
    dated["saledate"].str.split(" ").str[:4].str.join(" "),
    format="%a %b %d %Y", errors="coerce")

print("Parsed:", f"{dated['sale_dt'].notna().sum():,}",
      " Failed:", f"{dated['sale_dt'].isna().sum():,}")

dated = dated.dropna(subset=["sale_dt"])
print("Date range:", dated["sale_dt"].min().date(), "to", dated["sale_dt"].max().date())
print("Span:", (dated["sale_dt"].max() - dated["sale_dt"].min()).days, "days")

In [ ]:
weekly = dated.set_index("sale_dt").resample("W").agg(
    sales=("sellingprice", "size"),
    median_price=("sellingprice", "median"))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(weekly.index, weekly["sales"], color="#4C72B0")
axes[0].set_ylabel("Sales per week")
axes[0].set_title("Sales volume over time")

axes[1].plot(weekly.index, weekly["median_price"], color="#C44E52")
axes[1].set_ylabel("Median price ($)")
axes[1].set_xlabel("Sale date")
axes[1].set_title("Median selling price over time")

plt.tight_layout()
plt.show()

Two things stand out in the volume chart, and the first is a quirk of the data rather than a
finding. Sales are almost absent between early 2014 and late 2014, then jump sharply from December
2014 onward. The file is labeled as covering 2014 and 2015, and the dates do span 566 days, but the
bulk of the records sit in roughly the last eight months of that window. That matters for the split:
the training slice is dated from January 2014, but in practice most of its rows come from a much
narrower period. It does not break the chronological ordering, and it does not affect which sales
land in which split, but it is worth knowing that the effective training window is shorter than the
date range suggests.

The second is the price drift, and that one is a finding. Median weekly price moves over the period,
including a visible dip in spring 2015. This justifies two of our design decisions. It is why the
split has to be chronological rather than random, since a random split would let the model learn
from future sales to predict past ones, which inflates the score and could never happen in practice.
It is also the motivation for the adaptive conformal range in Will's half, because a prediction
interval calibrated once will stop covering correctly as the market moves underneath it.

## 9. Cleaning decisions

Pulling together what the sections above established.

In [ ]:
clean = cars.copy()
start = len(clean)
log = []

def drop(mask, reason):
    global clean
    n = int(mask.sum())
    clean = clean[~mask]
    log.append((reason, n, len(clean)))

# Target and benchmark must both exist.
drop(clean["sellingprice"].isna() | clean["mmr"].isna(), "missing sellingprice or mmr")

# A row with no make and no model describes nothing.
drop(clean["make"].isna() & clean["model"].isna(), "missing both make and model")

# Date is needed for the chronological split.
drop(clean["saledate"].isna(), "missing saledate")

# Implausible values. A price at or near zero is a data error, not a cheap car.
drop(clean["sellingprice"] <= 100, "selling price at or below $100")
drop(clean["mmr"] <= 100, "mmr at or below $100")
drop(clean["odometer"] > 500000, "odometer above 500,000 miles")

# Same physical car sold twice in the data.
dupes = clean.duplicated(subset=["vin", "saledate"], keep="first")
drop(dupes, "duplicate vin and saledate")

summary = pd.DataFrame(log, columns=["reason", "rows_removed", "rows_remaining"])
display(summary)
print(f"\nStarted with {start:,} rows, kept {len(clean):,} "
      f"({len(clean)/start:.1%}), removed {start-len(clean):,}")

In [ ]:
# Fill transmission blanks rather than dropping those rows, and parse the date.
clean["transmission"] = clean["transmission"].fillna("unknown")

clean["sale_dt"] = pd.to_datetime(
    clean["saledate"].str.split(" ").str[:4].str.join(" "),
    format="%a %b %d %Y", errors="coerce")
clean = clean.dropna(subset=["sale_dt"])

# Vehicle age at sale is more meaningful to a model than raw model year.
clean["age_at_sale"] = clean["sale_dt"].dt.year - clean["year"]
clean = clean[clean["age_at_sale"].between(-1, 50)]

print("Final cleaned shape:", clean.shape)
clean[["year", "age_at_sale", "odometer", "condition", "mmr", "sellingprice", "sale_dt"]].head()

Each removal is for a specific reason rather than a blanket `dropna()`. Rows missing the target or
the benchmark cannot be used or scored. Rows missing both make and model describe no identifiable
vehicle. Prices at or near zero are recording errors rather than genuinely free cars, and an
odometer above five hundred thousand miles is not credible.

Duplicate VIN and date pairs are removed because the same physical vehicle appearing twice in both
training and test would leak information across the split.

`transmission` blanks become an "unknown" category instead of causing a drop, since losing that
many rows would cost more than the column is worth.

The one feature I engineer here is `age_at_sale`, the difference between the sale year and the model
year. A model year of 2010 means something different in a 2014 sale than in a 2015 one, and age is
the quantity that actually drives depreciation.

## 10. Train, validation and test split

The split is chronological, 70 percent oldest for training, the next 15 percent for validation and
calibrating the prediction range, and the newest 15 percent held out for testing. This matches what
we described in the proposal.

A random split would be wrong here. The model would see sales from after the test period during
training, which is information no buyer has when bidding, and it would make the results look better
than they are.

In [ ]:
clean = clean.sort_values("sale_dt").reset_index(drop=True)

n = len(clean)
i_train = int(n * 0.70)
i_val = int(n * 0.85)

train = clean.iloc[:i_train]
val = clean.iloc[i_train:i_val]
test = clean.iloc[i_val:]

split_table = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(val), len(test)],
    "share": [f"{len(s)/n:.1%}" for s in (train, val, test)],
    "first_sale": [s["sale_dt"].min().date() for s in (train, val, test)],
    "last_sale": [s["sale_dt"].max().date() for s in (train, val, test)],
    "median_price": [f"${s['sellingprice'].median():,.0f}" for s in (train, val, test)],
    "mmr_mae": [f"${(s['sellingprice'] - s['mmr']).abs().mean():,.0f}"
                for s in (train, val, test)],
})
split_table

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.2))
for part, color, label in [(train, "#4C72B0", "train"),
                           (val, "#9467BD", "validation"),
                           (test, "#2E7D32", "test")]:
    ax.scatter(part["sale_dt"], np.random.uniform(0, 1, len(part)),
               s=1, alpha=0.05, color=color, label=label)
ax.set_yticks([])
ax.set_xlabel("Sale date")
ax.set_title("The split is chronological: the model is always tested on later sales")
leg = ax.legend(markerscale=12, loc="upper left")
for h in leg.legend_handles:
    h.set_alpha(1)
plt.tight_layout()
plt.show()

In [ ]:
# The benchmark to beat, measured on the test split only.
test_mae = (test["sellingprice"] - test["mmr"]).abs().mean()
test_medae = (test["sellingprice"] - test["mmr"]).abs().median()
test_mape = ((test["sellingprice"] - test["mmr"]).abs() / test["mmr"] * 100).mean()

print("MMR performance on the held-out test split")
print("=" * 44)
print(f"Vehicles:                 {len(test):,}")
print(f"Mean absolute error:      ${test_mae:,.0f}")
print(f"Median absolute error:    ${test_medae:,.0f}")
print(f"Mean percent error:       {test_mape:.1f}%")
print()
print("Any model we build has to beat these numbers on these same vehicles.")

## 11. What this data cannot tell us

Worth being direct about the boundaries before moving to modeling, since several of them shape what
we can honestly claim at the end.

The prices are from 2014 and 2015. Used car prices have moved a great deal since then, so the
dollar amounts here will not carry over to a sale happening today. What carries over is the method.
A dealer would retrain the same pipeline on their own recent purchases.

There is no profit information. The data records what a car sold for at auction, not what the
buyer later spent on reconditioning or what they resold it for. We can show that an estimate is
accurate. We cannot show that acting on it makes a dealer money. That is why our headline measure
is dollars of mispricing flagged rather than dollars earned.

There is no buyer in the data. The `seller` column identifies the consignor, not the dealer doing the
bidding. So we cannot study how individual buyers behave or whether any of them systematically
overpay.

No photographs are attached to these sales. The damage work in Will's half uses a separate
image dataset, which means for now we can test price and damage independently but cannot measure how
much a specific dent lowers a specific car's price. Connecting the two would need photographs
matched to the sale records, which neither dataset provides.

Nothing records how long a car took to sell. There is no listing duration or listing duration or
days on the lot, so we cannot say whether a vehicle will move quickly or sit. That rules out the
demand forecasting we considered earlier in planning.

None of these are modeling problems. They are all cases of data that does not exist in this source,
and naming them now keeps us from overclaiming later.

## 12. Dataset at a glance

A consolidated view of the numbers from the sections above.

In [ ]:
n_cat = sum(clean[c].nunique() > 2 for c in cat_cols if c in clean.columns)

glance = pd.DataFrame({
    "item": [
        "Dataset",
        "Source",
        "Samples, as downloaded",
        "Samples, after cleaning",
        "Features, original columns",
        "Features, after engineering",
        "Task type",
        "Target variable",
        "Number of classes",
        "Target range",
        "Target median",
        "Date range of sales",
        "Split method",
        "Training samples",
        "Validation samples",
        "Test samples",
        "Benchmark to beat (MMR on test)",
    ],
    "value": [
        "Vehicle Sales Data (wholesale auction sales)",
        "Kaggle, Syed Anwar Afridi (2024)",
        f"{len(cars):,}",
        f"{len(clean):,} ({len(clean)/len(cars):.1%} retained)",
        f"{cars.shape[1]}",
        f"{clean.shape[1]} (added sale_dt and age_at_sale)",
        "Regression",
        "sellingprice (US dollars)",
        "Not applicable, the target is continuous",
        f"${clean['sellingprice'].min():,.0f} to ${clean['sellingprice'].max():,.0f}",
        f"${clean['sellingprice'].median():,.0f}",
        f"{clean['sale_dt'].min().date()} to {clean['sale_dt'].max().date()}",
        "Chronological 70 / 15 / 15, no shuffling",
        f"{len(train):,}",
        f"{len(val):,}",
        f"{len(test):,}",
        f"${(test['sellingprice'] - test['mmr']).abs().mean():,.0f} mean absolute error",
    ],
})

display(glance.style.hide(axis="index"))

## 13. What this means for our three research questions

Our proposal set out three questions. This analysis does not answer any of them, since that is what
the models in stages 4 and 5 are for, but it does establish whether each one is worth asking and
what it will be measured against.

**Q1, price: can we predict where MMR will miss?** The analysis says the question is live. MMR is
unbiased in aggregate but misses by more than 1,000 dollars on 36.6 percent of cars, so there is
real error to attack. More importantly, that error is not random noise: it varies systematically by
price band, by condition grade, and by consignor. Section 7 is the strongest evidence, where rental
fleets and captive finance arms sell thousands of dollars under guide, consistently, across
thousands of sales. Structure like that is what a model can learn. The target to beat is now fixed
at 1,081 dollars mean absolute error on the held-out test sales.

**Q2, range: does a range that updates stay near 90 percent?** The analysis supports the premise.
Median weekly price moves over the period, including a visible dip in spring 2015, and the volume
pattern means our training window is effectively shorter than the date span suggests. Both make it
likely that an interval calibrated once will stop covering correctly on later sales, which is
exactly the comparison Will's half is built to run. The error distribution in section 6 is also wide
enough that a useful range will not be trivially narrow.

**Q3, damage: does pretraining help, and does it hold on real lot photos?** This dataset cannot
answer that, since it has no images. What it does contribute is the reason the question matters. MMR
is least accurate on cars at the rough end of the condition scale, where error runs about 1,396
dollars against 930 for cars in better shape. Condition is the variable the guide handles worst, and
it is the one thing a photograph can measure directly. That is the link between the two halves of
the project.

**And the measure we report to a dealer.** Our proposal promises a mispricing-caught number: the
dollars our flags would have warned a buyer about. Section 6 gives the raw material for it. On the
test split, the share of cars missing MMR by more than 1,000 dollars, multiplied by the size of
those misses, is the pool of money available to catch. Stage 5 reports how much of it our flags
actually find.

## 14. Summary and what this means for modeling

**The dataset.** Wholesale auction sales with 16 original columns covering what the vehicle is, its
condition and mileage, where and when it sold, the MMR guide value, and the price it actually
brought. After cleaning, the usable set keeps the large majority of the original rows.

**The target.** `sellingprice` in dollars. This is regression, not classification, so there are no
classes. The distribution is right skewed, which is why error gets reported both in dollars and as
a percentage, and broken out by price band.

**The split.** Chronological at 70 / 15 / 15. Training is the oldest sales, validation is the middle
slice and is also where the prediction range gets calibrated, and the newest sales are held out for
testing.

**What the analysis changed about our plan.**

The MMR error measured in section 6 is now our benchmark, and it is a real one rather than an
assumption. Every model in stage 4 and 5 gets scored against that number on the same test vehicles.

The cardinality table changed the encoding plan. `model`, `trim` and `seller` have far too many
distinct values for one-hot encoding, so the TensorFlow model will use learned embeddings for those
and one-hot encoding only for the small columns.

The nonlinear shape of the mileage and price relationship supports using a neural network over a
linear model, and gives the gradient boosting baseline a fair chance too since trees handle that
bend natively.

The price drift over time confirms the chronological split and supports the adaptive prediction
range, since an interval calibrated once will go stale as the market moves.

**Next step, stage 4.** Build the baseline price models on this cleaned and split data: MMR alone
as the benchmark, gradient boosting as the classical baseline, and the TensorFlow network with
embeddings as the deep learning model.

**Scope note.** This notebook covers the auction price data only. The CarDD damage images and the
photographs we collect at the auction are analyzed in Will's notebook in the same repository.

In [ ]:
# Save the cleaned splits so the stage 4 modeling notebook can load them directly.
os.makedirs("data", exist_ok=True)
for name, part in [("train", train), ("val", val), ("test", test)]:
    try:
        out = f"data/{name}.parquet"
        part.to_parquet(out, index=False)
    except Exception:
        # parquet needs pyarrow; fall back to compressed csv if it isn't installed
        out = f"data/{name}.csv.gz"
        part.to_csv(out, index=False, compression="gzip")
    print(f"{out}  {len(part):,} rows")

## References

Afridi, S. A. (2024). *Vehicle Sales Data* [Dataset]. Kaggle.
https://www.kaggle.com/datasets/syedanwarafridi/vehicle-sales-data

Manheim / Cox Automotive. *MMR Best Practices* and *MMR Help*.

Romano, Y., Patterson, E., & Candès, E. (2019). Conformalized quantile regression. *NeurIPS*.

Gibbs, I., & Candès, E. (2021). Adaptive conformal inference under distribution shift. *NeurIPS*.